In [ ]:
# Example 1: Basic ItemLoader with dict item using add_value and load_item.
from itemloaders import ItemLoader

loader = ItemLoader(item={})
loader.add_value('name', 'Alice')
loader.add_value('age', 30)
loader.add_value('tags', ['python', 'scraping'])

item = loader.load_item()
print(f"Loaded item: {item}")
assert item['name'] == ['Alice']
assert item['age'] == [30]
assert item['tags'] == ['python', 'scraping']


In [ ]:
# Example 2: Input and output processors via TakeFirst, MapCompose, Join.
from itemloaders import ItemLoader
from itemloaders.processors import TakeFirst, MapCompose, Join

class ProductLoader(ItemLoader):
    default_output_processor = TakeFirst()
    name_in = MapCompose(str.strip, str.title)
    tags_out = Join(', ')

loader = ProductLoader(item={})
loader.add_value('name', '  hello world  ')
loader.add_value('price', '19.99')
loader.add_value('tags', ['red', 'large', 'cotton'])

item = loader.load_item()
print(f"Item: {item}")
assert item['name'] == 'Hello World'
assert item['price'] == '19.99'
assert item['tags'] == 'red, large, cotton'


In [ ]:
# Example 3: ItemLoader with parsel Selector for HTML/XPath extraction.
from itemloaders import ItemLoader
from itemloaders.processors import TakeFirst
from parsel import Selector

html = '''
<html><body>
  <div class="product">
    <h1>Widget</h1>
    <span class="price">$9.99</span>
  </div>
</body></html>
'''
sel = Selector(text=html)
loader = ItemLoader(item={}, selector=sel)
loader.default_output_processor = TakeFirst()
loader.add_xpath('name', '//h1/text()')
loader.add_css('price', '.price::text')

item = loader.load_item()
print(f"Extracted: {item}")
assert item['name'] == 'Widget'
assert item['price'] == '$9.99'


In [ ]:
# Example 4: Identity, Compose, and SelectJmes processors.
from itemloaders.processors import Identity, Compose, SelectJmes, TakeFirst

identity = Identity()
assert identity([1, 2, 3]) == [1, 2, 3]

upper_first = Compose(lambda v: [s.upper() for s in v], TakeFirst())
assert upper_first(['hello', 'world']) == 'HELLO'

select = SelectJmes('foo.bar')
result = select({'foo': {'bar': 42}})
print(f"SelectJmes result: {result}")
assert result == 42


In [ ]:
# Example 5: replace_value, get_collected_values, nested loaders.
from itemloaders import ItemLoader
from itemloaders.processors import TakeFirst
from parsel import Selector

loader = ItemLoader(item={})
loader.add_value('color', 'red')
loader.add_value('color', 'blue')
assert loader.get_collected_values('color') == ['red', 'blue']

loader.replace_value('color', 'green')
assert loader.get_collected_values('color') == ['green']

# Nested loaders require a selector-backed parent (itemloaders API contract).
sel = Selector(text='<html><body><div id="a"><p>x</p></div></body></html>')
parent = ItemLoader(item={}, selector=sel)
parent.default_output_processor = TakeFirst()
parent.add_value('title', 'Outer')
nested = parent.nested_xpath('//div[@id="a"]')
nested.add_value('inner_field', 'inner_value')

final = parent.load_item()
print(f"Final item: {final}")
assert final['title'] == 'Outer'
assert final['inner_field'] == 'inner_value'